# **BLIP와 LLaVA**

## 1.환경준비

* 설치

In [ ]:
!pip install -U "transformers>=4.40,<4.48" accelerate pillow sentencepiece einops -q

* 라이브러리 로딩

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from transformers import pipeline
import torch
from PIL import Image

# 디바이스 준비
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## 2.BLIP 사용해보기

### (1) 이미지 캡셔닝

* 이미지 → 텍스트 캡션 파이프라인 준비

In [ ]:
cap_pipe = pipeline(
    task="image-to-text",
    model="Salesforce/blip-image-captioning-base",
    device=device
)

* 테스트 파일 준비

In [ ]:
from google.colab import files
uploaded = files.upload()   # 로컬 PC에서 cat_bird.jpg 선택

* 이미지 캡셔닝 테스트

In [ ]:
# 테스트용 이미지 경로
img_path = "cat_bird.jpg"   # 로컬에 있는 이미지 파일로 바꿔주세요

img = Image.open(img_path).convert("RGB")
plt.imshow(img)
plt.axis("off")
plt.show()

# 캡션 생성
cap = cap_pipe(img_path, max_new_tokens=30)[0]["generated_text"]
print("Caption:", cap)

### (2) Vision QA

* 시각 질의응답 모델 파이프라인

In [ ]:
vqa_pipe = pipeline(
    task="visual-question-answering",
    model="Salesforce/blip-vqa-base",
    device=device
)

* 테스트 파일 준비

In [ ]:
from google.colab import files
uploaded = files.upload()   # 로컬 PC에서 beach_children.jpg 선택

* VQA 테스트

In [ ]:
# VQA 질문 준비
img_path = "beach_children.jpg"
questions = ["What are they doing?"]

img = Image.open(img_path).convert("RGB")
plt.imshow(img)
plt.axis("off")
plt.show()

for q in questions:
    ans = vqa_pipe(image=img_path, question=q)[0]["answer"]
    print(f"Q: {q}\nA: {ans}\n")

### (3) 실습
* 이미지를 한장 업로드해서 이미지 캡셔닝 및 VQA를 수행해 봅니다.

In [ ]:
from google.colab import files
uploaded = files.upload()
img_path2 = "your_image.jpg"

* 이미지 캡셔닝

In [ ]:
img2 =

cap =
print("Caption:", cap)

* Vision QA

In [ ]:
# VQA 질문 준비
questions = []


for q in questions:
    ans =
    print(f"Q: {q}\nA: {ans}\n")

## 3.LLaVA 사용해보기

* tiny-llava 모델은 pipeline 함수로 지원 안됨

### (1) 모델 다운로드

In [ ]:
from transformers import AutoProcessor, LlavaForConditionalGeneration

device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if torch.cuda.is_available() else torch.float32

model_id = "bczhou/tiny-llava-v1-hf"

model = LlavaForConditionalGeneration.from_pretrained(
    model_id,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    low_cpu_mem_usage=True,
).to(device)

processor = AutoProcessor.from_pretrained(model_id)

### (2) 모델 사용

In [ ]:
img = Image.open("beach_children.jpg").convert("RGB")
prompt = "USER: <image>\nWhat do you see in this picture?\nASSISTANT:"   # <image> 꼭 포함!

# 전처리 → 생성 (인자 순서 주의: text 먼저, image 다음)
inputs = processor(prompt, img, return_tensors="pt").to(device)

with torch.inference_mode():
    output_ids = model.generate(**inputs, max_new_tokens=200, do_sample=False)

# 디코딩 (프롬프트 길이만큼 잘라서 답변만 출력)
answer = processor.decode(output_ids[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
print(answer)

### (3) 실습
* BLIP 실습에서 업로드한 이미지를 사용합니다.
* VQA를 위한 프롬프트를 작성하여 결과를 확인해 봅시다.

In [ ]:
prompt =

inputs = processor(prompt, img2, return_tensors="pt").to(device)

with torch.inference_mode():
    output_ids = model.generate(**inputs, max_new_tokens=200, do_sample=False)

answer = processor.decode(output_ids[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
print(answer)

## 4.BLIP와 LLaVA 비교

### (1) 테스트 이미지와 질문 준비

In [ ]:
# 이미 업로드한 파일 사용
img_path = "beach_children.jpg"
img = Image.open(img_path).convert("RGB")
questions = ["What is in the image?", "How many people are there?", "Describe the scene."]

### (2) 두 모델 비교

In [ ]:
#  BLIP 실행
print("=== BLIP (Salesforce/blip-vqa-base) ===================")
for q in questions:
    ans = vqa_pipe(img, q, max_new_tokens=32)[0]["answer"]
    print(f"Q: {q}\nA: {ans}\n")

#  LLaVA 실행
print("=== LLaVA (bczhou/tiny-llava-v1-hf) ===================")
for q in questions:
    prompt = f"USER: <image>\n{q}\nASSISTANT:"
    inputs = processor(prompt, img, return_tensors="pt").to(
        device,
        torch.float16 if torch.cuda.is_available() else torch.float32
    )
    with torch.inference_mode():
        out_ids = model.generate(**inputs, max_new_tokens=128, do_sample=False)
    answer = processor.decode(out_ids[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    print(f"Q: {q}\nA: {answer}\n")

### (3) 실습
새로운 이미지를 찾아서 비교해 봅시다

* 이미지 업로드 및 저장

In [ ]:
from google.colab import files
uploaded = files.upload()   # 로컬 PC에서 cat_bird.jpg 선택

img_path = "your image.jpg"



* 테스트 질문 저장

In [ ]:
questions = [    ]

* 모델 비교

In [ ]:
#  BLIP 실행
print("=== BLIP (Salesforce/blip-vqa-base) ===================")
for q in questions:
    ans = vqa_pipe(img, q, max_new_tokens=32)[0]["answer"]
    print(f"Q: {q}\nA: {ans}\n")

#  LLaVA 실행
print("=== LLaVA (bczhou/tiny-llava-v1-hf) ===================")
for q in questions:
    prompt = f"USER: <image>\n{q}\nASSISTANT:"
    inputs = processor(prompt, img, return_tensors="pt").to(
        device,
        torch.float16 if torch.cuda.is_available() else torch.float32
    )
    with torch.inference_mode():
        out_ids = model.generate(**inputs, max_new_tokens=128, do_sample=False)
    answer = processor.decode(out_ids[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    print(f"Q: {q}\nA: {answer}\n")